Dropout

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data ----
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
val_dataset   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)

# ---- Model with Dropout ----
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),      # <-- dropout after 1st hidden layer
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),      # <-- dropout after 2nd hidden layer
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(dropout_p=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ---- Train / Eval loops ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()          # enables dropout during training
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()            # disables dropout during evaluation
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ---- Run ----
num_epochs = 10
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

Epoch 1/10 | train_loss=0.3199 | val_loss=0.1300 | val_acc=0.9606
Epoch 2/10 | train_loss=0.1460 | val_loss=0.0855 | val_acc=0.9717
Epoch 3/10 | train_loss=0.1126 | val_loss=0.0824 | val_acc=0.9733
Epoch 4/10 | train_loss=0.0954 | val_loss=0.0784 | val_acc=0.9755
Epoch 5/10 | train_loss=0.0847 | val_loss=0.0739 | val_acc=0.9786
Epoch 6/10 | train_loss=0.0751 | val_loss=0.0669 | val_acc=0.9801
Epoch 7/10 | train_loss=0.0671 | val_loss=0.0664 | val_acc=0.9801
Epoch 8/10 | train_loss=0.0622 | val_loss=0.0705 | val_acc=0.9798
Epoch 9/10 | train_loss=0.0598 | val_loss=0.0645 | val_acc=0.9815
Epoch 10/10 | train_loss=0.0568 | val_loss=0.0635 | val_acc=0.9820


Normalization 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data ----
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
val_dataset   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)

# ---- Model with Dropout + BatchNorm ----
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),       # <-- batch norm before activation
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),       # <-- batch norm before activation
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(dropout_p=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ---- Train / Eval loops ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()          # enables dropout + uses batch statistics for BN
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()            # disables dropout + uses running mean/var for BN
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ---- Run ----
num_epochs = 10
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

Epoch 1/10 | train_loss=0.2809 | val_loss=0.1048 | val_acc=0.9673
Epoch 2/10 | train_loss=0.1330 | val_loss=0.0825 | val_acc=0.9727
Epoch 3/10 | train_loss=0.1057 | val_loss=0.0727 | val_acc=0.9784
Epoch 4/10 | train_loss=0.0879 | val_loss=0.0646 | val_acc=0.9784
Epoch 5/10 | train_loss=0.0751 | val_loss=0.0593 | val_acc=0.9805
Epoch 6/10 | train_loss=0.0691 | val_loss=0.0629 | val_acc=0.9781
Epoch 7/10 | train_loss=0.0604 | val_loss=0.0556 | val_acc=0.9821
Epoch 8/10 | train_loss=0.0574 | val_loss=0.0568 | val_acc=0.9832
Epoch 9/10 | train_loss=0.0535 | val_loss=0.0550 | val_acc=0.9829
Epoch 10/10 | train_loss=0.0490 | val_loss=0.0531 | val_acc=0.9831


Weight decay


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data ----
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
val_dataset   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)

# ---- Model with Dropout + BatchNorm ----
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(dropout_p=0.3).to(device)
criterion = nn.CrossEntropyLoss()

# ---- Weight Decay via optimizer ----
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4     # <-- L2 penalty on weights
)

# ---- Train / Eval loops ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ---- Run ----
num_epochs = 10
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

Epoch 1/10 | train_loss=0.2820 | val_loss=0.1066 | val_acc=0.9684
Epoch 2/10 | train_loss=0.1374 | val_loss=0.0824 | val_acc=0.9755
Epoch 3/10 | train_loss=0.1069 | val_loss=0.0718 | val_acc=0.9766
Epoch 4/10 | train_loss=0.0916 | val_loss=0.0673 | val_acc=0.9779
Epoch 5/10 | train_loss=0.0788 | val_loss=0.0664 | val_acc=0.9786
Epoch 6/10 | train_loss=0.0728 | val_loss=0.0612 | val_acc=0.9806
Epoch 7/10 | train_loss=0.0672 | val_loss=0.0567 | val_acc=0.9812
Epoch 8/10 | train_loss=0.0620 | val_loss=0.0583 | val_acc=0.9812
Epoch 9/10 | train_loss=0.0572 | val_loss=0.0586 | val_acc=0.9823
Epoch 10/10 | train_loss=0.0540 | val_loss=0.0553 | val_acc=0.9829


Early stoping 

In [3]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data ----
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
val_dataset   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)

# ---- Model with Dropout + BatchNorm ----
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(dropout_p=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ---- Train / Eval loops ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ---- Early Stopping helper ----
class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())  # save best weights
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

# ---- Run with Early Stopping ----
num_epochs = 50          # set high; early stopping will cut it short
early_stopper = EarlyStopping(patience=3, min_delta=1e-4)

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    early_stopper.step(val_loss, model)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1}. Best val_loss={early_stopper.best_loss:.4f}")
        break

# Restore best weights found during training
model.load_state_dict(early_stopper.best_state)

Epoch 1/50 | train_loss=0.2854 | val_loss=0.1100 | val_acc=0.9669
Epoch 2/50 | train_loss=0.1370 | val_loss=0.0852 | val_acc=0.9724
Epoch 3/50 | train_loss=0.1088 | val_loss=0.0687 | val_acc=0.9784
Epoch 4/50 | train_loss=0.0905 | val_loss=0.0699 | val_acc=0.9784
Epoch 5/50 | train_loss=0.0816 | val_loss=0.0661 | val_acc=0.9793
Epoch 6/50 | train_loss=0.0715 | val_loss=0.0600 | val_acc=0.9820
Epoch 7/50 | train_loss=0.0687 | val_loss=0.0603 | val_acc=0.9817
Epoch 8/50 | train_loss=0.0616 | val_loss=0.0619 | val_acc=0.9801
Epoch 9/50 | train_loss=0.0605 | val_loss=0.0550 | val_acc=0.9821
Epoch 10/50 | train_loss=0.0557 | val_loss=0.0561 | val_acc=0.9824
Epoch 11/50 | train_loss=0.0516 | val_loss=0.0542 | val_acc=0.9826
Epoch 12/50 | train_loss=0.0502 | val_loss=0.0547 | val_acc=0.9831
Epoch 13/50 | train_loss=0.0493 | val_loss=0.0553 | val_acc=0.9826
Epoch 14/50 | train_loss=0.0458 | val_loss=0.0571 | val_acc=0.9826
Early stopping triggered at epoch 14. Best val_loss=0.0542


<All keys matched successfully>

Learning rate sweep

In [4]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data ----
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
val_dataset   = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)

# ---- Model ----
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

criterion = nn.CrossEntropyLoss()

# ---- Train / Eval loops ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# ---- Early Stopping helper ----
class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

# ---- Single training run for a given LR ----
def run_training(lr, num_epochs=50, patience=3):
    model = MLP(dropout_p=0.3).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    early_stopper = EarlyStopping(patience=patience, min_delta=1e-4)

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        early_stopper.step(val_loss, model)
        if early_stopper.should_stop:
            break

    model.load_state_dict(early_stopper.best_state)
    final_val_loss, final_val_acc = evaluate(model, val_loader, criterion)
    return final_val_loss, final_val_acc, epoch + 1

# ---- LR Sweep ----
learning_rates = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
results = []

for lr in learning_rates:
    print(f"\n=== Training with lr={lr} ===")
    val_loss, val_acc, epochs_run = run_training(lr, num_epochs=50, patience=3)
    print(f"lr={lr} | final_val_loss={val_loss:.4f} | final_val_acc={val_acc:.4f} | epochs_run={epochs_run}")
    results.append({"lr": lr, "val_loss": val_loss, "val_acc": val_acc, "epochs_run": epochs_run})

# ---- Summary ----
print("\n=== LR Sweep Summary ===")
for r in sorted(results, key=lambda r: -r["val_acc"]):
    print(f"lr={r['lr']:<8} val_acc={r['val_acc']:.4f}  val_loss={r['val_loss']:.4f}  epochs={r['epochs_run']}")

best = max(results, key=lambda r: r["val_acc"])
print(f"\nBest LR: {best['lr']} (val_acc={best['val_acc']:.4f})")


=== Training with lr=0.0001 ===
lr=0.0001 | final_val_loss=0.0528 | final_val_acc=0.9834 | epochs_run=30

=== Training with lr=0.0003 ===
lr=0.0003 | final_val_loss=0.0536 | final_val_acc=0.9828 | epochs_run=22

=== Training with lr=0.001 ===
lr=0.001 | final_val_loss=0.0525 | final_val_acc=0.9832 | epochs_run=15

=== Training with lr=0.003 ===
lr=0.003 | final_val_loss=0.0585 | final_val_acc=0.9813 | epochs_run=15

=== Training with lr=0.01 ===
lr=0.01 | final_val_loss=0.0974 | final_val_acc=0.9703 | epochs_run=11

=== LR Sweep Summary ===
lr=0.0001   val_acc=0.9834  val_loss=0.0528  epochs=30
lr=0.001    val_acc=0.9832  val_loss=0.0525  epochs=15
lr=0.0003   val_acc=0.9828  val_loss=0.0536  epochs=22
lr=0.003    val_acc=0.9813  val_loss=0.0585  epochs=15
lr=0.01     val_acc=0.9703  val_loss=0.0974  epochs=11

Best LR: 0.0001 (val_acc=0.9834)
